### Yellow Dust Dataset - Preprocessing and EDA for V5 additional features

Contains China and Japan data. Yellow dust dataset from the Copernicus Atmosphere Monitoring Service.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import h3
from datetime import timezone

In [2]:
# === 1. Load Data ===
csv_path = "/home/julia/smoglens/data/yellow_dust_sfc.csv"
df = pd.read_csv(csv_path)

print("Initial shape:", df.shape)
print (df.head())
print ("Info: ")
df.info()

Initial shape: (6879672, 12)
            valid_time  latitude  longitude       u10       v10        t2m  \
0  2023-07-14 00:00:00      49.5      70.00  4.084112 -4.410157  292.92140   
1  2023-07-14 00:00:00      49.5      70.75  4.444952 -2.319825  293.97900   
2  2023-07-14 00:00:00      49.5      71.50  3.315069  0.891601  295.87450   
3  2023-07-14 00:00:00      49.5      72.25  1.098272  3.589354  297.51610   
4  2023-07-14 00:00:00      49.5      73.00 -0.573114  4.272948  297.07275   

   bcaod550  duaod550       lsm         pm2p5          pm10         sp  
0  0.007753  0.033607  0.989188  3.769855e-10  5.247784e-10  95463.766  
1  0.007878  0.046202  0.995354  1.591616e-12  2.273737e-12  95159.766  
2  0.007857  0.056740  0.994766  2.273737e-13  2.273737e-13  94840.766  
3  0.007964  0.062363  0.992191  9.906671e-10  1.457238e-09  94420.766  
4  0.008114  0.060853  0.990747  3.713240e-09  5.428092e-09  93657.766  
Info: 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6879672

## Reassign Columns

In [4]:
import h3
print(dir(h3))  # Should show 'geo_to_h3'

['H3BaseException', 'H3CellInvalidError', 'H3DirEdgeInvalidError', 'H3DomainError', 'H3DuplicateInputError', 'H3FailedError', 'H3GridNavigationError', 'H3LatLngDomainError', 'H3MemoryAllocError', 'H3MemoryBoundsError', 'H3MemoryError', 'H3NotNeighborsError', 'H3OptionInvalidError', 'H3PentagonError', 'H3ResDomainError', 'H3ResMismatchError', 'H3Shape', 'H3UndirEdgeInvalidError', 'H3ValueError', 'H3VertexInvalidError', 'LatLngMultiPoly', 'LatLngPoly', 'Literal', 'UnknownH3ErrorCode', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', '_cy', '_h3shape', '_version', 'api', 'are_neighbor_cells', 'average_hexagon_area', 'average_hexagon_edge_length', 'cell_area', 'cell_to_boundary', 'cell_to_center_child', 'cell_to_child_pos', 'cell_to_children', 'cell_to_children_size', 'cell_to_latlng', 'cell_to_local_ij', 'cell_to_parent', 'cell_to_vertex', 'cell_to_vertexes', 'cells_to_directed_edge', 'cells_to_geo', 'cell

In [ ]:
# === 2. Convert time to ISO 8601 in UTC ===
df['valid_time'] = pd.to_datetime(df['valid_time'], utc=True)
df['timestamp'] = df['valid_time'].dt.strftime('%Y-%m-%dT%H:%M:%SZ')

# === 3. Rename columns ===
df.rename(columns={
    'latitude': 'lat',
    'longitude': 'lon',
    'pm2p5': 'pm25',
    'pm10': 'pm10'
}, inplace=True)

# === 4. Generate H3 Index (Resolution 8) ===
def latlon_to_h3(lat, lon, res=8):
    return h3.geo_to_h3(lat, lon, res)

df['h3_index_res8'] = list(map(lambda lat, lon: h3.geo_to_h3(lat, lon, 8), df['lat'], df['lon']))


# Add H3 center coordinates
df['h3_lat_res8'] = df['h3_index_res8'].apply(lambda x: h3.h3_to_geo(x)[0])
df['h3_lon_res8'] = df['h3_index_res8'].apply(lambda x: h3.h3_to_geo(x)[1])

# === 5. Add metadata ===

df['data_source'] = 'cds'   # Climate Data Store

# Assign country based on longitude: JP (Japan) for lon >= 120, CN (China) for lon < 120
def assign_country(row):
    if row['lon'] >= 120:
        return 'JP'
    else:
        return 'CN'

df['country'] = df.apply(assign_country, axis=1)

# === 6. Filter columns ===
columns_to_keep = [
    'timestamp', 'h3_index_res8', 'h3_lat_res8', 'h3_lon_res8',
    'pm25', 'pm10', 'u10', 'v10', 't2m', 'bcaod550', 'duaod550', 'sp', 'data_source', 'country'
]
df = df[columns_to_keep]


: 